# Notebook 10 -- Time Patterns

## 1. Load Data

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 2. Starbucks Filter

In [2]:
df   = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
sbux = df[df["brand_category"] == "Starbucks"]
print(f"Starbucks reviews: {len(sbux):,}")

Starbucks reviews: 11,675


## 3. Day-of-Week Ratings

In [3]:
DOW_LABELS = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}

dow = (
    sbux.groupby("day_of_week")
    .agg(
        Reviews      =("review_id",   "count"),
        Avg_Stars    =("review_stars", "mean"),
        Pct_Critical =("star_tier",   lambda x: round((x == "Critical").mean() * 100, 1)),
    )
    .round({"Avg_Stars": 2})
    .reset_index()
)
dow["Day"] = dow["day_of_week"].map(DOW_LABELS)
dow.columns = ["dow_num", "Reviews", "Avg Stars", "% Critical", "Day"]
display(dow[["Day", "Reviews", "Avg Stars", "% Critical"]])

,Day,Reviews,Avg Stars,% Critical
0,Mon,1557,2.90,47.3
1,Tue,1530,3.02,45.3
2,Wed,1520,3.07,42.6
3,Thu,1649,3.03,43.8
4,Fri,1691,2.95,45.9
5,Sat,1873,2.72,52.7
6,Sun,1855,2.70,53.6


## 4. Day-of-Week Chart

In [4]:
display(dow[["Day", "Avg Stars", "% Critical"]])

# Weekend bars in red, weekday bars in Starbucks green
bar_colors = ["#d62728" if d in ["Sat", "Sun"] else "#00704A" for d in dow["Day"]]

fig = go.Figure(go.Bar(
    x=dow["Day"],
    y=dow["Avg Stars"],
    marker_color=bar_colors,
    text=[f"{v:.2f}" for v in dow["Avg Stars"]],
    textposition="outside",
))
# Reference line: overall Starbucks avg
overall_avg = sbux["review_stars"].mean()
fig.add_hline(
    y=overall_avg,
    line_dash="dot",
    line_color="#888888",
    annotation_text=f"Overall avg {overall_avg:.2f}",
    annotation_position="top right",
)
fig.update_layout(
    title=dict(text="Starbucks — Avg Star Rating by Day of Week", font=dict(size=16)),
    xaxis_title="Day of Week",
    yaxis_title="Avg Star Rating",
    plot_bgcolor="white",
    paper_bgcolor="white",
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", range=[0, 4.0]),
    xaxis=dict(showgrid=False),
    width=700, height=420,
    margin=dict(t=60, b=50, l=60, r=40),
)
fig.write_html(str(FIGURES_DIR / "10_dow_rating.html"))
fig.show()

,Day,Avg Stars,% Critical
0,Mon,2.90,47.3
1,Tue,3.02,45.3
2,Wed,3.07,42.6
3,Thu,3.03,43.8
4,Fri,2.95,45.9
5,Sat,2.72,52.7
6,Sun,2.70,53.6


## 5. Weekday vs. Weekend Comparison

In [5]:
wk = (
    sbux.groupby("is_weekend")
    .agg(
        Reviews      =("review_id",    "count"),
        Avg_Stars    =("review_stars",  "mean"),
        Avg_Sentiment=("sentiment_score","mean"),
        Pct_Critical =("star_tier",    lambda x: round((x == "Critical").mean() * 100, 1)),
        Pct_Positive =("star_tier",    lambda x: round((x == "Positive").mean() * 100, 1)),
    )
    .round({"Avg_Stars": 2, "Avg_Sentiment": 3})
    .reset_index()
)
wk["Period"] = wk["is_weekend"].map({False: "Weekday (Mon–Fri)", True: "Weekend (Sat–Sun)"})
wk = wk[["Period", "Reviews", "Avg_Stars", "Avg_Sentiment", "Pct_Critical", "Pct_Positive"]]
wk.columns = ["Period", "Reviews", "Avg Stars", "Avg Sentiment", "% Critical", "% Positive"]
display(wk)

,Period,Reviews,Avg Stars,Avg Sentiment,% Critical,% Positive
0,Weekday (Mon–Fri),7947,2.99,0.304,45.0,44.7
1,Weekend (Sat–Sun),3728,2.71,0.219,53.2,37.3


## 6. Monthly Trend

In [6]:
MONTH_LABELS = {
    1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
    7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec",
}
monthly = (
    sbux.groupby("month")
    .agg(
        Reviews  =("review_id",    "count"),
        Avg_Stars=("review_stars", "mean"),
    )
    .round({"Avg_Stars": 2})
    .reset_index()
)
monthly["Month"] = monthly["month"].map(MONTH_LABELS)

display(monthly[["Month", "Reviews", "Avg_Stars"]])

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=monthly["Month"],
    y=monthly["Avg_Stars"],
    mode="lines+markers",
    line=dict(color="#00704A", width=2.5),
    marker=dict(color="#00704A", size=8),
    name="Starbucks",
))
fig2.add_hline(
    y=overall_avg,
    line_dash="dot",
    line_color="#888888",
    annotation_text=f"Overall avg {overall_avg:.2f}",
    annotation_position="top right",
)
fig2.update_layout(
    title=dict(text="Starbucks — Avg Star Rating by Month (2017–2021 Combined)", font=dict(size=16)),
    xaxis_title="Month",
    yaxis_title="Avg Star Rating",
    plot_bgcolor="white",
    paper_bgcolor="white",
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", range=[2.5, 3.5]),
    xaxis=dict(showgrid=False),
    width=800, height=400,
    margin=dict(t=60, b=50, l=60, r=80),
)
fig2.write_html(str(FIGURES_DIR / "10_monthly_rating.html"))
fig2.show()

,Month,Reviews,Avg_Stars
0,Jan,1000,3.01
1,Feb,899,3.02
2,Mar,1034,2.99
3,Apr,863,2.92
4,May,931,2.79
5,Jun,973,2.94
6,Jul,989,2.88
7,Aug,971,2.91
8,Sep,1011,2.78
9,Oct,943,2.87


## Key Findings

- Weekends average 2.71 stars vs. weekday 2.99 stars, a 0.28-star gap. 53.2% of weekend reviews are critical vs. 45.0% on weekdays.
- Wednesday peaks at 3.07 stars, followed by Thursday (3.03) and Tuesday (3.02).
- Sunday is the worst day at 2.70 stars with 53.6% critical reviews.
- September (2.78) and May (2.79) are the lowest-rated months. January and February are highest at 3.01-3.02 stars.